# Whisper Paraphasia Detection — Training

Fine-tune Whisper on Fridriksson data with LOSO cross-validation.

**Runtime:** Change to A100 GPU via Runtime > Change runtime type.

In [1]:
# Check GPU
!nvidia-smi

Tue Mar 24 21:00:35 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   30C    P0             45W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## 1. Setup

In [ ]:
# Mount Google Drive (for persistent storage of checkpoints + data)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone repo (or pull latest)
import os

REPO_DIR = '/content/aphasia-modeling'
DRIVE_DIR = '/content/drive/MyDrive/aphasia-modeling'

# Create drive dir for persistent data
os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(f'{DRIVE_DIR}/checkpoints', exist_ok=True)

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    # TODO: replace with your repo URL
    !git clone https://github.com/YOUR_USERNAME/aphasia-modeling.git {REPO_DIR}

In [ ]:
# Install dependencies
!cd {REPO_DIR} && pip install -e '.[dev]' wandb -q

In [ ]:
# Set wandb API key
import os
os.environ['WANDB_API_KEY'] = ''  # <-- paste your key here

## 2. Upload Data

Upload `fridriksson.json` and the `audio/` folder to Google Drive at:
```
My Drive/aphasia-modeling/data/fridriksson.json
My Drive/aphasia-modeling/data/audio/*.wav
```

Or upload directly to the Colab runtime (faster I/O but ephemeral).

In [ ]:
# Option A: Copy data from Drive to local (faster I/O during training)
DATA_DIR = f'{REPO_DIR}/data'
os.makedirs(f'{DATA_DIR}/Fridriksson/audio', exist_ok=True)

!cp {DRIVE_DIR}/data/fridriksson.json {DATA_DIR}/fridriksson.json
!cp {DRIVE_DIR}/data/audio/*.wav {DATA_DIR}/Fridriksson/audio/

# Verify
!ls -la {DATA_DIR}/fridriksson.json
!ls {DATA_DIR}/Fridriksson/audio/ | wc -l
print('audio files found ^')

In [ ]:
# Update audio paths in the dataset to point to local copies
import json
from pathlib import Path

data_path = Path(f'{DATA_DIR}/fridriksson.json')
records = json.loads(data_path.read_text())

audio_dir = Path(f'{DATA_DIR}/Fridriksson/audio')
updated = 0
for r in records:
    if r.get('audio_path'):
        # Rewrite path to point to local audio dir
        wav_name = Path(r['audio_path']).name
        new_path = str(audio_dir / wav_name)
        if Path(new_path).exists():
            r['audio_path'] = new_path
            updated += 1

data_path.write_text(json.dumps(records, indent=2))
print(f'Updated {updated}/{len(records)} audio paths')

## 3. Train — Single Fold (test first)

Run one fold first to make sure everything works before committing to full LOSO.

In [ ]:
!cd {REPO_DIR} && python scripts/train.py \
    --phase 2 \
    --data_path data/fridriksson.json \
    --model_name openai/whisper-small \
    --test_speaker fridriksson01 \
    --output_dir {DRIVE_DIR}/checkpoints/fold_fridriksson01 \
    --class_weights \
    --bf16 \
    --num_workers 2 \
    --wandb

## 4. Train — Full LOSO (all 13 folds)

Only run this after the single fold above succeeds. ~8-13 hours on A100.

In [ ]:
!cd {REPO_DIR} && python scripts/train.py \
    --phase 2 \
    --data_path data/fridriksson.json \
    --model_name openai/whisper-small \
    --loso \
    --output_dir {DRIVE_DIR}/checkpoints/loso \
    --class_weights \
    --bf16 \
    --num_workers 2 \
    --wandb

## 5. Evaluate

In [ ]:
# Evaluate a single fold
!cd {REPO_DIR} && python scripts/evaluate_model.py \
    --model_path {DRIVE_DIR}/checkpoints/fold_fridriksson01 \
    --data_path data/fridriksson.json \
    --test_speaker fridriksson01 \
    --output_dir {DRIVE_DIR}/results/fold_fridriksson01